In [24]:
# !python -m pip install lightning

import numpy as np
import pandas as pd

import os

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from pathlib import Path


from torchvision import datasets, transforms
from torchvision.transforms import v2


import pytorch_lightning as pl
from pytorch_lightning import Trainer, callbacks
from pytorch_lightning.loggers import TensorBoardLogger
import datetime


print(os.getcwd())

C:\Users\flash\Desktop\Jobvorbereitungen\Projekte\DataScience\ImageClassification


In [25]:
import logging
import smtplib
import os
os.makedirs("Logs", exist_ok= True)
use_mail = False

logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

formatter = logging.Formatter('%(name)s : %(levelname)s:%(levelno)s  %(asctime)s    %(message)s ')

info_handler = logging.FileHandler('Logs/Advanced_Logging_INFO_Level.log')
info_handler.setLevel(logging.INFO)
info_handler.setFormatter(formatter)
logger.addHandler(info_handler)



if use_mail:
    from email_logger_config import email_info
    login = "your@gmail.com"
    pswd = "**** **** **** ****" # Configured over gmail app-password
    email_info_example = {
    "mailhost": ('smtp.gmail.com', 587), # Everywhere on teh internet it recommended port 465, but what ended working was port 587
    "fromaddr": ["your@gmail.com"],
    "toaddrs": ["your@gmail.com"],
    "subject": 'Script Error Message',
    "credentials": (login, pswd),
    "secure": (),
    "timeout": 15.0 # time-out  default is 1 second, which leads to a time-out error
    }

    error_emailer = logging.handlers.SMTPHandler(**email_info)

    error_emailer.setLevel(logging.ERROR)
    logger.addHandler(error_emailer)
else:
    error_handler = logging.FileHandler('Logs/Advanced_Logging_ERROR_Level.log')
    error_handler.setLevel(logging.ERROR)
    error_handler.setFormatter(formatter)
    logger.addHandler(error_handler)

# Hypothetical


logger.info("Loggers created")

In [26]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    accelerator = "gpu"
    from google.colab import drive
    drive.mount('/content/drive')
    root_dir = "./drive/MyDrive/Colab Data/ResidualNeuralNetwork"
    torch.cuda.memory.empty_cache()
else:
    device = torch.device("cpu")
    accelerator = "cpu"
    root_dir = "."

In [27]:
!set "PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True"

In [28]:
class ImageFolderCustom(Dataset):
    def __init__(self, targ_dir, transform=None):
        self.dirs = os.listdir(targ_dir) # each folder represents one class and contains the images of that class
        self.paths = []
        self.classes = []
        for i, Dir in enumerate(self.dirs):
            paths = list(Path(f"{targ_dir}/{Dir}").glob("*.jpg"))
            self.paths.extend(paths)
            self.classes.extend([i] * len(paths))
        self.transform = transform
        self.cache = {}
        
    def load_image(self, index):
        image_path = self.paths[index]
        return Image.open(image_path)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        if not (index in self.cache):
            img = self.load_image(index)
            class_idx = self.classes[index]
            self.cache[index] = (img, class_idx)
        else:
            img, class_idx = self.cache[index]

        if self.transform:
            return self.transform(img), class_idx
        else:
            return img, class_idx

In [29]:
# prepare the Data and DataAugmentation
batch_size = 75
height = 128
width = 128
classes = 2
shuffle = True
valid_size = 0.1
pin_memory = True if torch.cuda.is_available() else False
num_workers = 0
persistent_workers = False
##### Detemine mean and std for each channel ##########################################
Basic_Augmentation_Pipeline = v2.Compose([
    v2.ToImage(), v2.ToDtype(torch.float32, scale=True), # v2.ToTensor() is apparently depricatd and this new line is the replacement
    v2.Resize((width, height))])

train_val_dataset_prep = ImageFolderCustom(f"{root_dir}/Data/Cats_Dogs/train", transform = Basic_Augmentation_Pipeline) # datasets.ImageFolder

num_train = len(train_val_dataset_prep)
indices = list(range(num_train))
split = int(np.floor(valid_size * num_train))

if shuffle:
    np.random.seed(79)
    np.random.shuffle(indices)

train_idx, valid_idx = indices[split:], indices[:split]
train_sampler = torch.utils.data.sampler.SubsetRandomSampler(train_idx)
valid_sampler = torch.utils.data.sampler.SubsetRandomSampler(valid_idx)


train_loader = DataLoader(train_val_dataset_prep, batch_size=batch_size, sampler=train_sampler)
samples = 0
means = torch.tensor([0.0, 0.0, 0.0])
stds = torch.tensor([0.0, 0.0, 0.0])
for img in next(iter(train_loader)):
    if len(img.shape) == 1:
        continue
    samples += img.shape[0]
    img = img.view(img.shape[0], img.shape[1], -1)
    batch_means = img.mean(dim=2).sum(dim = 0)
    batch_std = img.std(dim=2).sum(dim = 0)
    means += batch_means
    stds += batch_std

means /= samples
stds /= samples

Image_augmentation_pipeline = v2.Compose([
    v2.ToImage(), v2.ToDtype(torch.float32, scale=True), # v2.ToTensor() is apparently depricatd and this new line is the replacement
    v2.RandomHorizontalFlip(p = 0.5),
    v2.RandomVerticalFlip(p= 0.5),
    v2.RandomApply([v2.RandomCrop(30)], p = 0.5),
    v2.Resize((width, height)),
    v2.Normalize(means, stds)
])

Validation_augmentation_pipeline = Image_augmentation_pipeline = v2.Compose([
    v2.ToImage(), v2.ToDtype(torch.float32, scale=True), # v2.ToTensor() is apparently depricatd and this new line is the replacement
    v2.Resize((width, height)),
    v2.Normalize(means, stds)
])

train_dataset = ImageFolderCustom(f"{root_dir}/Data/Cats_Dogs/train", transform = Image_augmentation_pipeline) # datasets.ImageFolder
train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=train_sampler, num_workers = num_workers, pin_memory = pin_memory, persistent_workers = persistent_workers)
val_dataset = ImageFolderCustom(f"{root_dir}/Data/Cats_Dogs/train", transform = Validation_augmentation_pipeline)
valid_loader = DataLoader(val_dataset, batch_size=batch_size, sampler=valid_sampler, num_workers = num_workers, pin_memory = pin_memory, persistent_workers = persistent_workers)

# Currently the labels are indices range(0, classes). In the pytorch lightnign class, the labels will be one-hot encoded before training
# In Integers: 0 is Cat, 1 is Dog.
# In OneHot: [1, 0] is Cat, [0, 1] is Dog.

In [30]:
# Start TensorBoard server once trainign has started over the anaconda prompter
###### tensorboard --logdir=Checkpoints

# Open browser to http://localhost:6006
# View real-time loss curves, histograms, and model graphs

In [31]:
def shrinkage(In, kernel_size, stride, padding):
    return int(((In + 2*padding - 1* (kernel_size-1) -1)/stride)+1)

In [32]:
# Build and Compile the Model
from torch.optim.lr_scheduler import ReduceLROnPlateau, StepLR


class ConvBlock(nn.Module):
    def __init__(self, in_features, out_features, stride = 1, activ = "Tanh", kernel_size = 3, padding = 1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_features, in_features, kernel_size = kernel_size, stride = 1, padding = padding)
        self.conv2 = nn.Conv2d(in_features, out_features, kernel_size = kernel_size, stride = stride, padding = padding)

        if activ == "Tanh":
            self.activ_f = nn.Tanh()
        elif activ == "SELU":
            self.activ_f = nn.SELU()
        
        for m in [self.conv1, self.conv2]:
            if activ == "Tanh":
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)
            elif activ == "SELU":
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='selu')
                
        
        self.norm1 = nn.BatchNorm2d(in_features)
        self.norm2 = nn.BatchNorm2d(out_features)
        self.dropout = nn.Dropout(0.2)

        if in_features == out_features and stride == 1:
            self.skip = nn.Identity()
        else:
            self.skip = nn.Sequential(
                nn.Conv2d(in_features, out_features, kernel_size = kernel_size, stride = stride, padding = padding),
                nn.BatchNorm2d(out_features)
            )
            for m in [self.conv1, self.conv2]:
                if isinstance(m, nn.Conv2d):
                    if activ == "Tanh":
                        nn.init.xavier_uniform_(m.weight)
                        nn.init.zeros_(m.bias)
                    elif activ == "ReLU":
                        nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')

    def forward(self, x):
        skip = self.skip(x)

        x = self.conv1(x)
        x = self.norm1(x)
        x = self.activ_f(x)
        x = self.dropout(x)

        x = self.conv2(x)
        x = self.norm2(x) + skip
        x = self.activ_f(x)
        x = self.dropout(x)
        return x


class Linear_Block(nn.Module):
    def __init__(self, in_nodes, out_nodes, activ = "SELU"):
        super().__init__()
        self.fc1 = nn.Linear(in_nodes, in_nodes)
        self.fc2 = nn.Linear(in_nodes, out_nodes)

        self.skip = nn.Identity() if in_nodes == out_nodes else nn.Linear(in_nodes, out_nodes)

        if activ == "SELU":
            self.activ_f = nn.SELU()
            for layer in [self.fc1, self.fc2, self.skip]:
                if isinstance(layer, nn.Linear):
                    nn.init.kaiming_uniform_(layer.weight, mode='fan_in', nonlinearity='selu')
        elif activ == "Tanh":
            self.activ_f = nn.Tanh()
            for layer in [self.fc1, self.fc2, self.skip]:
                if isinstance(layer, nn.Linear):
                    nn.init.xavier_uniform_(layer.weight)
                    nn.init.zeros_(layer.bias)

        self.norm1 = nn.BatchNorm1d(in_nodes)
        self.norm2 = nn.BatchNorm1d(out_nodes)
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        skip = self.skip(x)
        x = self.fc1(x)
        x = self.norm1(x)
        x = self.activ_f(x)
        x = self.dropout(x)

        x = self.fc2(x)
        x = self.norm2(x) + skip
        x = self.activ_f(x)
        x = self.dropout(x)
        return x
        
        

class ConvNet_Lightning(pl.LightningModule):
    def __init__(self, hp):
        """
        Build the network. 
        """
        super().__init__()
        self.hp = hp
        self.num_classes = hp.num_classes
        # 128*128
        self.conv1 = ConvBlock(3, 32, stride = 2, activ = "SELU") # 64*64
        self.conv2 = ConvBlock(32, 64, stride = 2, activ = "SELU") # 32 * 32
        self.conv3 = ConvBlock(64, 128, stride = 2, activ = "SELU") # 16*16
        self.avg_pool = nn.AvgPool2d(4, stride=4) # 4*4
        self.flatten = lambda batch_size, x: x.reshape(batch_size, -1)

        self.fc1 = Linear_Block(128*4*4, 512, activ = "SELU")
        self.fc2 = Linear_Block(512, 64, activ = "SELU")
        self.out_layer = nn.Linear(64, self.num_classes)

        self.loss = torch.nn.CrossEntropyLoss(label_smoothing = hp.label_smoothing)


    def _base(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.avg_pool(x)
        x = self.flatten(x.shape[0], x)
        x = self.fc1(x)
        x = self.fc2(x)
        y_pred = self.out_layer(x)
        return y_pred

    def configure_optimizers(self):
        # "weigth decay" is L2 Regularizer
        optimizer = torch.optim.Adam(self.parameters(), lr= self.hp.lr, weight_decay = self.hp.l2)# , momentum = 0.9, nesterov = True)
        scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=hp.lr_decay_gamma, patience=hp.lr_decay_patience, threshold = hp.lr_decay_threshold)
        return  {"optimizer": optimizer, "lr_scheduler": scheduler, "monitor": "train_loss"}
        # scheduler = StepLR(optimizer, gamma = hp.lr_decay_gamma, step_size = hp.lr_decay_patience)
        # return {"optimizer": optimizer, "lr_scheduler": scheduler}
        

    def training_step(self, batch, batch_idx):
        """
        Implement a single training period with loss function. Training loss is logged (by default to Tensorboard)
        """
        x, y = batch
        y_pred = self._base(x)

        loss = self.loss(y_pred, y)
        y_pred = torch.argmax(y_pred.detach(), dim = 1).flatten()
        acc = np.mean((y_pred == y.flatten()).numpy().astype(np.int32))
        # Logging to TensorBoard (if installed) by default
        self.log("train_loss", loss)
        self.log("train_acc", acc)
        return loss

    def validation_step(self, batch, batch_idx):
        """
        Implement a single validation period with loss function. Validation loss is logged (by default to Tensorboard)
        """
        x, y = batch
        y_pred = self._base(x)
        loss = self.loss(y_pred, y)
        y_pred = torch.argmax(y_pred.detach(), dim = 1).flatten()
        acc = np.mean((y_pred == y.flatten()).numpy().astype(np.int32))
        acc_just0 = np.mean((np.zeros(shape = y_pred.shape) == y.flatten().numpy()).astype(np.int32))
        acc_just1 = np.mean((np.ones(shape = y_pred.shape) == y.flatten().numpy()).astype(np.int32))
        # Logging to TensorBoard (if installed) by default
        # Logging to TensorBoard (if installed) by default
        self.log("val_loss", loss)
        self.log("val_acc", acc)
        self.log("val_acc_just_0", acc_just0)
        self.log("val_acc_just_1", acc_just1)

    def test_step(self, batch, batch_idx):
        """
        Implement a single validation period with loss function. Validation loss is logged (by default to Tensorboard)
        """
        x, y = batch
        y_pred = self._base(x)

        loss = self.loss(y_pred, y)
        y_pred = torch.argmax(y_pred.detach(), dim = 1).flatten()
        acc = np.mean((y_pred == y.flatten()).numpy().astype(np.int32))
        # Logging to TensorBoard (if installed) by default
        # Logging to TensorBoard (if installed) by default
        self.log("test_loss", loss)
        self.log("test_acc", acc)


    def forward(self, x, convert_to_string = False):
        """
        For prediction after training
        """
        cut = False
        with torch.no_grad():
            if x.shape[0] == 1:
                x = torch.concatenate((x, x), dim = 0)
                cut = True
            y_pred = self._base(x)
            y_pred = np.argmax(y_pred.numpy(), axis = 1)

            if convert_to_string:
                y_pred = ["Cat" if i == 0 else "Dog" for i in y_pred]

            if cut:
                return [y_pred[0]]
            else:
                return y_pred

        
    

In [33]:
os.makedirs(f"{root_dir}/Models/Pytorch", exist_ok = True)

In [34]:
class Hyperparameters:
    epochs = 100
    lr = 1e-3
    l2 = 1e-4
    lr_decay_gamma = 0.1
    lr_decay_patience = 5
    lr_decay_threshold = 1e-3
    num_classes = classes
    earlystop_vallos = 1e-4
    earlystop_patience = 20
    gradient_clip_val = 1.0
    label_smoothing = 0.4
    # accumulate_grad_batches = 30

    ####### 85% Validierung
    # lr = 1e-3
    # label_smooting = 0.3
    # ReduceLROnPlateau(..., gamma = 0.1, waiting = 5, threshold = 1e-3), looking on train_loss
    # No Colorjitter
    # RandomApply(Cop(100), p = 0.5)

hp = Hyperparameters()

In [ ]:
from lightning.pytorch.loggers import CSVLogger
if not "BestModel.save" in  os.listdir(f"{root_dir}/Models/Pytorch"):
    # Only do all of this when the Finished Model has not yet been generated
    
    earlyStop = callbacks.EarlyStopping(monitor='val_loss', patience=hp.earlystop_patience, min_delta = hp.earlystop_vallos)
    # checkpPointTraining: Saves the weights per epoch to continue training in case of interruptions.
    checkPointTraining = callbacks.ModelCheckpoint(dirpath = f"{root_dir}/Checkpoints", filename = "Pyboard_Test_{epoch}.weights", monitor='val_loss', verbose=0, save_top_k = -1, save_weights_only=False)
    trainer = Trainer(callbacks=[earlyStop, checkPointTraining],
                      logger = CSVLogger("Logs", "CSVLogger"),
                      max_epochs = hp.epochs,
                      accelerator = accelerator,
                      gradient_clip_val = hp.gradient_clip_val,
                      # accumulate_grad_batches = hp.accumulate_grad_batches
                 )
    # Pyboard_Test_{epoch}.weights.ckpt
    SavePoints = [i for i in os.listdir(f"{root_dir}/Checkpoints") if i.startswith("Pyboard_Test") and i.endswith(".ckpt")]
    try:
        # Do checkpoints alerady exist ?
        initial_epoch = np.max([int(i.split(".")[0].split("=")[-1]) for i in SavePoints])
        
    except:
        initial_epoch = 0

    print(initial_epoch)

    # Either Start trainign from Scratch if no checkpoints exist or continue training from checkpoint
    ConvNet = ConvNet_Lightning(hp = hp).to(device)
    if initial_epoch == 0:
        # load training weights
        trainer.fit(model = ConvNet, train_dataloaders = train_loader, val_dataloaders = valid_loader)
    else:
        trainer.fit(model = ConvNet, train_dataloaders = train_loader, val_dataloaders = valid_loader, ckpt_path=f"{root_dir}/Checkpoints/Pyboard_Test_epoch={initial_epoch}.weights.ckpt")

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
Restoring states from the checkpoint path at ./Checkpoints/Pyboard_Test_epoch=10.weights.ckpt


10


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ conv1     │ ConvBlock        │  2.0 K │ train │     0 │
│ 1 │ conv2     │ ConvBlock        │ 46.6 K │ train │     0 │
│ 2 │ conv3     │ ConvBlock        │  185 K │ train │     0 │
│ 3 │ avg_pool  │ AvgPool2d        │      0 │ train │     0 │
│ 4 │ fc1       │ Linear_Block     │  6.3 M │ train │     0 │
│ 5 │ fc2       │ Linear_Block     │  329 K │ train │     0 │
│ 6 │ out_layer │ Linear           │    130 │ train │     0 │
│ 7 │ loss      │ CrossEntropyLoss │      0 │ train │     0 │
└───┴───────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 6.9 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 6.9 M                                                                                                
Total estimated model params size (MB): 27.452                                                                     
Modules in train mode: 49                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Restored all states from the checkpoint at ./Checkpoints/Pyboard_Test_epoch=10.weights.ckpt


Output()

In [ ]:
import os
import shutil
if "BestModel.save" not in os.listdir(f"{root_dir}/Models/Pytorch"):
    ConvNet = ConvNet_Lightning.load_from_checkpoint(checkPointTraining.best_model_path, hp = hp, weights_only = False)
    saving_Trainer = Trainer(max_epochs = 0)
    saving_Trainer.fit(ConvNet, train_dataloaders = train_loader, val_dataloaders = valid_loader)
    saving_Trainer.save_checkpoint(f"{root_dir}/Models/Pytorch/BestModel.save", weights_only = True)
ConvNet = ConvNet_Lightning.load_from_checkpoint(f"{root_dir}/Models/Pytorch/BestModel.save", hp = hp, weights_only = True)

In [ ]:
import glob
from PIL import Image

class TestDataset(torch.utils.data.Dataset):
    def __init__(self, path, transform=None):
        self.image_paths = glob.glob(path + '*.jpg')
        self.transform = transform

    def __getitem__(self, index):
        x = Image.open(self.image_paths[index])
        if self.transform is not None:
            x = self.transform(x)

        return x

    def __len__(self):
        return len(self.image_paths)

test_dataset = TestDataset(f"{root_dir}/Data/Cats_Dogs/test1/", transform = Validation_augmentation_pipeline)
test_loader = torch.utils.data.DataLoader(test_dataset, shuffle = True)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
stop = 50
fig = plt.figure(figsize = (20, 20))
for i, image in enumerate(test_loader):
    label = ConvNet.forward(image, convert_to_string = True)
    rows = stop//10
    columns = stop//rows
    ax = fig.add_subplot(rows, columns, i+1)
    # PyTorch uses (channels, width, height) for images, but matplotlib uses (width, height, channels). Use .permute to change it from PyTorch to Matplotlib format

    image_for_plotting = image.reshape(image.shape[1], image.shape[2], image.shape[3]).permute(1, 2, 0)
    ax.imshow(image_for_plotting*stds + means)
    ax.set_title(label)
    

    if i == stop-1:
        break
plt.show()
    

In [ ]:
37/50